# 引言

在第一个notebook中，使用了application数据集构建，为了提高分数，引入更多数据特征

手动特征工程是繁琐的，通常依赖领域知识。
我们尽可能注入更多特征， 由模型识别。
我们会也会使用一些自动化特征工具，
会使用pca等降维

会大量使用pandas操作
- groupby
- agg 对分组计算
- merge 汇总
- rename 列
- 

会涉及到很多特征阶段。 我们通过feather命名区分：
命名规则：`[阶段]_[来源表]_[处理程度]`
- `01_app_train_raw.feather` : 01原始， 主表，； 仅做了onehot类型编
- `02_prev_app_agg.feather`: 02子表聚合， ； agg特征
- `03_bureau_cleaned.feather`: 03子表 ： 特征选择剔除：missing、高corr
- `	04_merged_v1.feather` : 04 合并子表：
- `04_main_bureau_prev_combined.feather` 

这里，第一部分中 讨论引入`bureau` `bureau balance`

# 导入包

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.style.use('fivethirtyeight')

# 第一部分 

第一部分主要做两件事：
- 引入bureau和bureau_balance特征，描述用户之前在homecredit贷款情况
- 描述了手动特征的一般工程
  - 分类特征：
  - id特征： count计数
  - 数值统计特征：   

## bureau
- 产出 bureau_agg

In [ ]:
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
print(train.shape, test.shape)

In [ ]:
bureau = pd.read_csv('data/bureau.csv')
bureau.shape


### SK_ID_BUREAU计数
- 产生了`bureau_previous_loan_counts`特征集

In [ ]:
bureau_previous_loan_counts = bureau.groupby(by='SK_ID_CURR', as_index= False)['SK_ID_BUREAU'].count().rename(columns = {
    'SK_ID_BUREAU': 'bureau_previous_loan_counts'
})

In [ ]:
bureau_previous_loan_counts.sort_values(by='bureau_previous_loan_counts', ascending=False)

In [ ]:
bureau_previous_loan_counts.shape

#### Assessing Usefulness of New Variable with r value

In [ ]:
train[['SK_ID_CURR','TARGET']]

In [ ]:
bureau_previous_loan_counts_plot = pd.merge(train[['SK_ID_CURR','TARGET']], bureau_previous_loan_counts, on='SK_ID_CURR', how='left')
bureau_previous_loan_counts_plot = bureau_previous_loan_counts_plot.fillna(0)
bureau_previous_loan_counts_plot

In [ ]:
sns.kdeplot(
    data= bureau_previous_loan_counts_plot,
    x = 'bureau_previous_loan_counts',
    hue = 'TARGET',
    common_norm = False
)
plt.title('bureau_previous_loan_counts distribution')


几乎是同分布的，没什么区别

In [ ]:
corr = np.corrcoef(bureau_previous_loan_counts_plot['bureau_previous_loan_counts'],
     bureau_previous_loan_counts_plot['TARGET'])[0, 1]
print(f'correlations bureau_previous_loan_counts and TARGET is {corr}')

In [ ]:
del bureau_previous_loan_counts_plot
gc.collect()

相关性很低，没有进入之前的top5. 我们没有得到什么信息。

### Agg the numeric volumns
- 我们产生了`bureau_numeric_agg` 新的特征集

In [ ]:
bureau.dtypes

In [ ]:
bureau_numeric = bureau.select_dtypes(exclude=['str'])

In [ ]:
bureau_numeric_agg = bureau_numeric.drop(columns=['SK_ID_BUREAU']).groupby(by='SK_ID_CURR', as_index=False).agg(
    ['min','mean',  'max', 'sum']
)
bureau_numeric_agg

我们把多层索引平铺

In [ ]:
bureau_numeric_agg.columns = [
    f"bureau_{col[0]}_{col[1]}" if col[1] != "" else col[0] 
    for col in bureau_numeric_agg.columns.values
]
bureau_numeric_agg

In [ ]:
bureau_numeric_agg.shape

计算一下相关性把

In [ ]:
new_columns = list(bureau_numeric_agg.columns)
new_columns.remove('SK_ID_CURR')
new_columns

In [ ]:
corrs = bureau_numeric_agg[new_columns].corrwith(train['TARGET']).sort_values(ascending=False)
corrs.head()

In [ ]:
corrs.tail()

可以看到，我们构造出几个正相关 还不错的特征！！！😍

继续画图看看

In [ ]:
print(train.shape, bureau_numeric_agg.shape)

In [ ]:
bureau_numeric_agg_plot = pd.merge(train[['SK_ID_CURR', 'TARGET']], 
    bureau_numeric_agg[['SK_ID_CURR', 'bureau_DAYS_CREDIT_mean']],
    on = 'SK_ID_CURR',
    how = 'left'
    )

In [ ]:
plt.figure(figsize=(5,3))
sns.kdeplot(
    data = bureau_numeric_agg_plot,
    x = 'bureau_DAYS_CREDIT_mean',
    hue = 'TARGET',
    common_norm=False
)

可以看到，分布还是有点区别，那些`DAYS_CREDIT_mean` 平均贷款天数更多的越容易违约

### categorical columns
`bureau_categorical_agg`

对于分类变量， 我们可以统计次数和平均次数

In [ ]:
bureau_categorical = pd.get_dummies(bureau.select_dtypes(include='str'))
bureau_categorical['SK_ID_CURR'] = bureau['SK_ID_CURR']
bureau_categorical

In [ ]:
bureau_categorical_agg = bureau_categorical.groupby(by='SK_ID_CURR').agg(
    ['sum', 'mean']
)
bureau_categorical_agg.head()

In [ ]:
bureau_categorical_agg.columns[0]

In [ ]:
bureau_categorical_agg.columns = [ f'bureau_{col[0]}_{col[1]}' for col in bureau_categorical_agg.columns]
bureau_categorical_agg.head()

In [ ]:
bureau_categorical_agg = bureau_categorical_agg.reset_index()
bureau_categorical_agg.head()

把上述特征合并起来

In [ ]:
print(bureau_categorical_agg.shape, bureau_previous_loan_counts.shape, bureau_numeric_agg.shape)

In [ ]:
dfs = [df.set_index('SK_ID_CURR') for df in [bureau_categorical_agg, bureau_previous_loan_counts, bureau_numeric_agg]]
bureau_agg = pd.concat(dfs, axis=1)
bureau_agg = bureau_agg.reset_index()
bureau_agg.head()

In [ ]:
bureau_agg.shape

In [ ]:
bureau_agg.to_feather('checkpoints/02_bureau_agg.feather')

In [ ]:
del bureau_categorical_agg, bureau_previous_loan_counts, bureau_numeric_agg,bureau_agg
gc.collect()

## bureau_balance
- 产生 `bureau_balance_by_client_agg`

In [ ]:
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')

In [ ]:
bureau_balance = pd.read_csv('data/bureau_balance.csv')
bureau_balance.shape

In [ ]:
bureau_balance.head()

In [ ]:
bureau_balance.dtypes

In [ ]:
bureau_balance.head()

因此对于`MONTHS_BALANCE` 可以聚合数字特征。`STATUS` 为分类统计。

这是很多相同的。我们可以为此写两个函数`agg_numeric` `agg_categorical`

In [ ]:
def agg_numeric(df : pd.DataFrame, group_column, df_name = '', exclude_columns = []):
    """ 聚合数值特征: ['min', 'max', 'mean', 'sum'] 这是一般共有的
    group_column:
    df_name:
    exclude_columns: 需要排除一些id列. 一般不需要。
    """
    numeric_df = df.select_dtypes('number')
    numeric_df = numeric_df.drop(columns = exclude_columns)

    numeric_df[group_column] = df[group_column]
    numeric_df_agg = numeric_df.groupby(by = group_column).agg(
        ['min', 'max', 'mean', 'sum']
    )
    prefix = f'{df_name}_' if df_name != '' else ''
    numeric_df_agg.columns = [
        f"{prefix}{col[0]}_{col[1]}".upper()
        for col in numeric_df_agg.columns.values
    ]
    numeric_df_agg = numeric_df_agg.reset_index()
    return numeric_df_agg

In [ ]:
def agg_categorical(df : pd.DataFrame, group_column, df_name=''):
    """ 聚合数值特征: ['mean', 'sum'] 这是一般共有的
    group_column:
    df_name:
    """
    categorical_df = pd.get_dummies(df.select_dtypes(include = ['str', 'object', 'category']))
    categorical_df[group_column] = df[group_column]
    categorical_df_agg = categorical_df.groupby(by = group_column).agg(
        ['mean', 'sum']
    )
    categorical_df_agg.columns =  [
        f"{df_name}_{col[0]}_{col[1]}" if col[1] != "" else col[0] 
        for col in categorical_df_agg.columns.values
    ]
    categorical_df_agg = categorical_df_agg.reset_index()
    return categorical_df_agg

In [ ]:
bureau_balance_numeric_agg = agg_numeric(bureau_balance, 'SK_ID_BUREAU', 'bureau', exclude_columns=['SK_ID_BUREAU'])
bureau_balance_numeric_agg.head()

In [ ]:
bureau_balance_numeric_agg.shape

In [ ]:
bureau_balance_categorical_agg = agg_categorical(bureau_balance, 'SK_ID_BUREAU', 'bureau')
bureau_balance_categorical_agg.head()

In [ ]:
bureau_balance_categorical_agg.shape

In [ ]:
bureau_balance_agg = pd.merge(bureau_balance_numeric_agg, bureau_balance_categorical_agg, on='SK_ID_BUREAU', how='left')
bureau_balance_agg.head()

### 进一步聚合到SK_ID_CURR

In [ ]:
bureau[['SK_ID_CURR', 'SK_ID_BUREAU']]

In [ ]:
bureau_balance_by_client = pd.merge(
    bureau_balance_agg,
    bureau[['SK_ID_CURR', 'SK_ID_BUREAU']], on='SK_ID_BUREAU', how='right'
    )

In [ ]:
bureau_balance_by_client.head()

这里id-bureau 会有多个，因为每个用户之前有多个申请记录， 我们聚合一下

In [ ]:
bureau_balance_by_client_agg = agg_numeric(bureau_balance_by_client, 'SK_ID_CURR', '', exclude_columns=['SK_ID_CURR', 'SK_ID_BUREAU'])

In [ ]:
bureau_balance_by_client_agg.head()

由于一对多，所以行数增加的

In [ ]:
bureau_balance_by_client_agg.to_feather('checkpoints/02_bureau_balance_agg.feather')

## 特征选择

In [ ]:
bureau_balance_agg = pd.read_feather('checkpoints/02_bureau_balance_agg.feather')
bureau_agg = pd.read_feather('checkpoints/02_bureau_agg.feather')

train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
print(train.shape, test.shape)

train = pd.merge(train, bureau_agg, on='SK_ID_CURR', how='left')
train = pd.merge(train, bureau_balance_agg, on='SK_ID_CURR', how='left')
test = pd.merge(test, bureau_agg, on='SK_ID_CURR', how='left')
test = pd.merge(test, bureau_balance_agg, on='SK_ID_CURR', how='left')

print(train.shape, test.shape)

### 缺失值

In [ ]:
def missing_values_table(df):
    """ 统计缺失值
    """
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * df.isnull().sum() / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table_ren_columns = mis_val_table.rename(columns= {0:'Missing Values', 1:'% of total values'})
    mis_val_table_ren_columns = mis_val_table_ren_columns.sort_values(by='% of total values', ascending = False)
    mis_val_table_ren_columns = mis_val_table_ren_columns.loc[mis_val_table_ren_columns['% of total values'] != 0, :]

    return mis_val_table_ren_columns

In [ ]:
missing_train = missing_values_table(train)
missing_train.head(10)

对于一些缺失过多的字段可以考虑删除掉。比如去掉 超过90%的列

In [ ]:
missing_90_columns = missing_train.index[missing_train['% of total values'] > 90]

In [ ]:
missing_90_columns

In [ ]:
missing_values_table(test)

### 对齐train和test列

In [ ]:
print(train.shape, test.shape)

In [ ]:
train_labels = train['TARGET']
train, test = train.align(test, join='inner', axis=1)
train['TARGET'] = train_labels

In [ ]:
print(train.shape, test.shape)

### 对这个特征工程看看相关性， 是不是更好呢？

In [ ]:
%%time
corrs = train.corr()

In [ ]:
corrs_sorted = corrs.sort_values(by='TARGET', ascending=False)

In [ ]:
corrs_sorted['TARGET'].head(10)

In [ ]:
corrs_sorted['TARGET'].tail(10)

可以看到，我们的一些新特征确实有更好的相关性

In [ ]:
sns.kdeplot(
    data = train,
    x = 'bureau_CREDIT_ACTIVE_Active_mean',
    hue = 'TARGET',
    common_norm = False
)

好吧，目前看来没啥用

此外我们可以剔除一些高度相关变量

In [ ]:
corrs.head()

In [ ]:
corr_abs = corrs.abs()

In [ ]:
threshold = 0.8
high_corr = {}
for col in corr_abs:
    high_corr[col] = list(corr_abs.index[(corr_abs[col] > threshold) & (corr_abs[col] != 1)])
high_corr

对于高度相关的特征对.
- 如果

In [ ]:
upper = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
    )   
upper

In [ ]:
high_corr_pairs = upper.unstack().reset_index()
high_corr_pairs.columns = ['feature_1', 'feature_2', 'corr']
high_corr_pairs = high_corr_pairs[high_corr_pairs['corr'] > threshold]
high_corr_pairs

In [ ]:
to_drop = set()
missing_counts = train.isnull().sum()
for index, row in high_corr_pairs.iterrows():
    f1, f2 = row['feature_1'], row['feature_2']
    if f1 in to_drop or f2 in to_drop: # 有一个已经在drop了
        continue
    if missing_counts[f1] > missing_counts[f2]:
        to_drop.add(f1)
    else:
        to_drop.add(f2)
to_drop = list(to_drop)
to_drop

从训练集和测试集移除这些列，

In [ ]:
len(to_drop)

In [ ]:
train.columns

In [ ]:
train_corrs_removed = train.drop(columns=to_drop)
test_corrs_removed = test.drop(columns=to_drop)

In [ ]:
train_corrs_removed.to_feather('checkpoints/04_train_app_bureau_balance_bureau_cleaned.feather')
test_corrs_removed.to_feather('checkpoints/04_test_app_bureau_balance_bureau_cleaned.feather')

In [ ]:
del corrs, corr_abs, train_corrs_removed, test_corrs_removed
gc.collect()

## modeling

### 导入
- 每个model运行前，建议重新导入一次

In [ ]:
train = pd.read_feather('checkpoints/04_train_app_bureau_balance_bureau_cleaned.feather')
test = pd.read_feather('checkpoints/04_test_app_bureau_balance_bureau_cleaned.feather')

train_labels = train['TARGET']
train_ids = train['SK_ID_CURR']
test_ids = test['SK_ID_CURR']
train_features = train.drop(columns=['TARGET', 'SK_ID_CURR'])
test_features = test.drop(columns=['SK_ID_CURR'])

In [ ]:
print(train.shape, test.shape)

### HistGradientBoostingClassifier

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

HistGradientBoostingClassifier 
- 不需要处理缺失值
- 树模型对量级不敏感，不需要scaler

In [ ]:
%%time
hist_gradient_boost_model= HistGradientBoostingClassifier(
    max_iter = 100, # 树个数
    learning_rate = 0.1,
    max_depth = 5,
)
hist_gradient_boost_model.fit(train_features, train_labels)

In [ ]:
from sklearn.metrics import roc_auc_score
train_prob = hist_gradient_boost_model.predict_proba(train_features)
train_prob

In [ ]:
from sklearn.metrics import  roc_curve
fpr, tpr, thresholds = roc_curve(train_labels, train_prob[:, 1])
auc = roc_auc_score(train_labels, train_prob[:, 1])

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(fpr, tpr, color='blue', lw=2)
plt.title(f'hist gb Roc curve, auc={auc:.3f}')

In [ ]:
hist_gradient_boost_model_pred = hist_gradient_boost_model.predict_proba(test_features)

In [ ]:
submit = pd.DataFrame({
    'SK_ID_CURR': test_ids
})
submit['TARGET'] = hist_gradient_boost_model_pred[:, 1]

submit.to_csv('hist_gradient_boost_model_with_bureau.csv', index = False)
submit.head()

In [ ]:
submit.shape

得分73

### lightgbm
- 需要清理列名

In [ ]:
import re
# 1. 定义清理函数
def clean_names(df):
    # 替换所有非字母、数字的字符为下划线
    # 这里的正则 [^A-Za-z0-9_] 会匹配空格、斜杠、括号等所有特殊字符
    df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]
    # 顺便处理一下可能出现的重复下划线，比如 __
    df.columns = [re.sub(r'_+', '_', col).strip('_') for col in df.columns]
    return df

In [ ]:
train_features = clean_names(train_features)
test_features = clean_names(test_features)

In [ ]:
%%time
from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(
    n_estimators=100,      # 对应 max_iter，树的个数
    learning_rate=0.1,     # 学习率
    max_depth=3,           # 树的最大深度
    random_state=42,       # 保证结果可复现
    n_jobs=-1              # 使用所有 CPU 核心加速
)
lgbm_model.fit(train_features, train_labels)

In [ ]:
lgbm_model_pred = lgbm_model.predict_proba(test_features)[:, 1]

In [ ]:
submit = pd.DataFrame(
    {
        'SK_ID_CURR': test_ids
    }
)
submit['TARGET'] = lgbm_model_pred

submit.to_csv('lgbm_model_pred_with_bureau.csv', index = False)

得分 73

In [ ]:
features_importance = pd.DataFrame(
    {
        'importance': lgbm_model.feature_importances_,
        'feature': lgbm_model.feature_name_
    }
)

In [ ]:
features_importance_plot = features_importance.sort_values(by='importance', ascending=False).head(20)
features_importance_plot

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data = features_importance_plot,
    x= 'importance',
    y = 'feature'
)
plt.tight_layout()

还是有点提升的，我们看到有一些新的重要特征

# 第二部分
- 按照第一部分步骤，做一些最基本的处理。使用`previous_application 、 POS_CASH_balance 、 installments_payments 和 credit_card_balance`文件

In [ ]:
def get_missing_columns(df, rate=90):
    """只计算需要删除的列名"""
    missing_stats = df.isnull().sum() / len(df) * 100
    to_drop = missing_stats[missing_stats > rate].index.tolist()
    return to_drop

In [ ]:
def get_high_corr_columns(df, threshold=0.9):
    """
    高效获取高相关特征，优先保留缺失值较少的特征
    """
    # 1. 计算相关性矩阵
    corr_matrix = df.corr().abs()
    
    # 2. 提取上三角（不含对角线）
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # 3. 找出所有超过阈值的列名
    # 这里的 to_drop 是我们要剔除的特征候选名单
    to_drop = set()
    
    # 4. 获取缺失值统计
    missing_counts = df.isnull().sum()
    
    # 5. 遍历每一列，检查是否存在高相关
    for column in upper.columns:
        # 找到与当前列 column 相关性大于阈值的所有特征
        high_corr_features = upper[column][upper[column] > threshold].index.tolist()
        
        for feature in high_corr_features:
            # 比较 column 和 feature 的缺失值情况
            # 谁缺失多删谁
            if missing_counts[column] > missing_counts[feature]:
                to_drop.add(column)
                break # column 既然要被删了，就不用再看它与其他特征的关系了
            else:
                to_drop.add(feature)
                
    return list(to_drop)


In [ ]:
def feature_select(train, test):
    """ 移除 高缺失值列和高相关特征
    """ 
    train = train.copy()
    test = test.copy()

    train_labels = train['TARGET']
    train_ids = train['SK_ID_CURR']
    test_ids = test['SK_ID_CURR']

    # 这两列不参与
    train = train.drop(columns=['TARGET', 'SK_ID_CURR'])
    test = test.drop(columns=['SK_ID_CURR'])

    train = train.drop(columns=get_missing_columns(train))
    test = test.drop(columns=get_missing_columns(test))
    print('remove high missing cols. ', train.shape, test.shape)

    train, test = train.align(test, join='inner', axis=1)
    print('align train and test.', train.shape, test.shape)
    
    train_sample = train.sample(n=int(len(train) * 0.3))
    to_drop_columns = get_high_corr_columns(train_sample)
    train = train.drop(columns=to_drop_columns)
    test = test.drop(columns=to_drop_columns)
    
    train['TARGET'] = train_labels
    train['SK_ID_CURR'] = train_ids
    test['SK_ID_CURR'] = test_ids

    print('remove high corr cols.', train.shape, test.shape)

    return train, test

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import  roc_curve

def plot_roc(targets, prob, name):
    fpr, tpr, thresholds = roc_curve(targets, prob)
    auc = roc_auc_score(targets, prob)
    plt.figure(figsize=(3,3))
    plt.plot(fpr, tpr, color='blue', lw=2)
    plt.title(f'{name} Roc curve, auc={auc:.3f}')

## 引入previous_application表
- previous_agg_by_client

In [ ]:
previous = pd.read_csv('data/previous_application.csv')


In [ ]:
previous.head()

In [ ]:
previous.shape

In [ ]:
previous.columns

In [ ]:
previous.dtypes

In [ ]:
previous_categorical_agg = agg_categorical(previous, ['SK_ID_PREV', 'SK_ID_CURR'], 'previous')

In [ ]:
previous_categorical_agg.head()

In [ ]:
previous_numeric_agg = agg_numeric(previous,['SK_ID_PREV', 'SK_ID_CURR'], 'previous')

In [ ]:
previous_numeric_agg.head()

In [ ]:
previous_agg = pd.merge(previous_numeric_agg, previous_categorical_agg, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')
previous_agg.head()

按照client聚合

In [ ]:
previous_agg_by_client = agg_numeric(previous_agg, 'SK_ID_CURR', exclude_columns=['SK_ID_PREV'])

In [ ]:
previous_agg_by_client.columns

In [ ]:
previous_agg_by_client.head()

In [ ]:
sk_id_prev_cnts = previous_agg.groupby(by='SK_ID_CURR')['SK_ID_PREV'].count().reset_index().rename(columns = {'SK_ID_PREV' : 'prev_applications_counts'})

In [ ]:
sk_id_prev_cnts

In [ ]:
previous_agg_by_client = pd.merge(
    previous_agg_by_client,
    sk_id_prev_cnts,
    on = 'SK_ID_CURR',
    how = 'left'
)

In [ ]:
previous_agg_by_client

In [ ]:
previous_agg_by_client.to_feather('checkpoints/02_previous_agg.feather')

In [ ]:
del previous, previous_categorical_agg, previous_numeric_agg, previous_agg,sk_id_prev_cnts
gc.collect()

特征选择

In [ ]:
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
previous_agg = pd.read_feather('checkpoints/02_previous_agg.feather')

train = pd.merge(train, previous_agg, on='SK_ID_CURR', how='left')
test = pd.merge(test, previous_agg, on='SK_ID_CURR', how='left')

print(f'train: {train.shape}, test: {test.shape}')
train, test = feature_select(train, test)
print(f'train: {train.shape}, test: {test.shape}')

train.to_feather('checkpoints/04_train_app_previous_cleaned.feather')
test.to_feather('checkpoints/04_test_app_previous_cleaned.feather')


## credit_card_balance

In [ ]:
credit_card_balance = pd.read_csv('data/credit_card_balance.csv')

In [ ]:
credit_card_balance.head()

In [ ]:
credit_card_balance.dtypes

In [ ]:
credit_card_balance_numeric_agg = agg_numeric(credit_card_balance, ['SK_ID_CURR', 'SK_ID_PREV'], 'credit_card_balance')

In [ ]:
credit_card_balance_numeric_agg

In [ ]:
credit_card_balance_categorical_agg = agg_categorical(credit_card_balance, ['SK_ID_CURR', 'SK_ID_PREV'], 'credit_card_balance')

In [ ]:
credit_card_balance_categorical_agg

In [ ]:
credit_card_balance_agg = pd.merge(credit_card_balance_numeric_agg, credit_card_balance_categorical_agg, 
    on = ['SK_ID_CURR', 'SK_ID_PREV'],
    how = 'left'
)

In [ ]:
credit_card_balance_agg_by_client = agg_numeric(credit_card_balance_agg, 'SK_ID_CURR', exclude_columns=['SK_ID_PREV'])

In [ ]:
credit_card_balance_agg_by_client.head()

In [ ]:
credit_card_balance_agg_by_client.to_feather('checkpoints/02_credit_balance_agg.feather')

In [ ]:
del credit_card_balance, credit_card_balance_numeric_agg, credit_card_balance_categorical_agg, credit_card_balance_agg
gc.collect()

In [ ]:
print(train.shape, test.shape)

In [ ]:
missing_values_table(train)

In [ ]:
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
credit_card_balance_agg = pd.read_feather('checkpoints/02_credit_balance_agg.feather')

train = pd.merge(train, credit_card_balance_agg, on='SK_ID_CURR', how='left')
test = pd.merge(test, credit_card_balance_agg, on='SK_ID_CURR', how='left')

print(f'train: {train.shape}, test: {test.shape}')
train, test = feature_select(train, test)
print(f'train: {train.shape}, test: {test.shape}')

train.to_feather('checkpoints/04_train_app_credit_cleaned.feather')
test.to_feather('checkpoints/04_test_app_credit_cleaned.feather')

In [ ]:
del credit_card_balance_agg
gc.collect()

## 引入 pos_cash_balance表

In [ ]:
pos_cash_balance = pd.read_csv('data/pos_cash_balance.csv')

In [ ]:
pos_cash_balance_numeric_agg = agg_numeric(pos_cash_balance, ['SK_ID_CURR', 'SK_ID_PREV'], 'pos')

In [ ]:
pos_cash_balance_numeric_agg

In [ ]:
pos_cash_balance_categorical_agg = agg_categorical(pos_cash_balance, ['SK_ID_CURR', 'SK_ID_PREV'], 'pos')

In [ ]:
pos_cash_balance_categorical_agg

In [ ]:
pos_cash_balance_agg = pd.merge(pos_cash_balance_categorical_agg, pos_cash_balance_numeric_agg, 
    on = ['SK_ID_CURR', 'SK_ID_PREV'],
    how = 'left'
)

In [ ]:
pos_cash_balance_agg_by_client = agg_numeric(pos_cash_balance_agg, 'SK_ID_CURR', exclude_columns=['SK_ID_PREV'])

In [ ]:
pos_cash_balance_agg_by_client.to_feather('checkpoints/02_pos_agg.feather')

In [ ]:
del pos_cash_balance, pos_cash_balance_numeric_agg, pos_cash_balance_categorical_agg, pos_cash_balance_agg
gc.collect()

In [ ]:
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
pos_cash_balance_agg = pd.read_feather('checkpoints/02_pos_agg.feather')

train = pd.merge(train, pos_cash_balance_agg, on='SK_ID_CURR', how='left')
test = pd.merge(test, pos_cash_balance_agg, on='SK_ID_CURR', how='left')

print(f'train: {train.shape}, test: {test.shape}')
train, test = feature_select(train, test)
print(f'train: {train.shape}, test: {test.shape}')

train.to_feather('checkpoints/04_train_app_pos_cleaned.feather')
test.to_feather('checkpoints/04_test_app_pos_cleaned.feather')

## installments_payments表

In [ ]:
installments_payments = pd.read_csv('data/installments_payments.csv')

In [ ]:
installments_payments.dtypes

没有分类特征，都是数值的

In [ ]:
installments_payments_numeric_agg = agg_numeric(installments_payments, ['SK_ID_CURR', 'SK_ID_PREV'], 'installments')

In [ ]:
installments_payments_agg = installments_payments_numeric_agg

In [ ]:
installments_payments_agg_by_client = agg_numeric(installments_payments_agg, 'SK_ID_CURR', exclude_columns=['SK_ID_PREV'])

In [ ]:
installments_payments_agg_by_client.head()

In [ ]:
del installments_payments, installments_payments_numeric_agg, installments_payments_agg
gc.collect()

In [ ]:
installments_payments_agg_by_client.to_feather('checkpoints/02_installments_agg.feather')

In [ ]:
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
installments_payments_agg = pd.read_feather('checkpoints/02_installments_agg.feather')

train = pd.merge(train, installments_payments_agg, on='SK_ID_CURR', how='left')
test = pd.merge(test, installments_payments_agg, on='SK_ID_CURR', how='left')

print(f'train: {train.shape}, test: {test.shape}')
train, test = feature_select(train, test)
print(f'train: {train.shape}, test: {test.shape}')

train.to_feather('checkpoints/04_train_app_installments_cleaned.feather')
test.to_feather('checkpoints/04_test_app_installments_cleaned.feather')

## 合并所以app-子表

In [ ]:
# 仅查看appbase shape
train = pd.read_feather('checkpoints/01_train_app_base.feather')
test = pd.read_feather('checkpoints/01_test_app_base.feather')
print('app', train.shape, test.shape)

In [ ]:
train_app_previous = pd.read_feather('checkpoints/04_train_app_previous_cleaned.feather')
test_app_previous = pd.read_feather('checkpoints/04_test_app_previous_cleaned.feather')
print('app_previous', train_app_previous.shape, test_app_previous.shape)

train_app_credit = pd.read_feather('checkpoints/04_train_app_credit_cleaned.feather')
test_app_credit= pd.read_feather('checkpoints/04_test_app_credit_cleaned.feather')
print('app_credit', train_app_credit.shape, test_app_credit.shape)

train_app_pos = pd.read_feather('checkpoints/04_train_app_pos_cleaned.feather')
test_app_pos = pd.read_feather('checkpoints/04_test_app_pos_cleaned.feather')
print('app_pos', train_app_pos.shape, test_app_pos.shape)

train_app_install = pd.read_feather('checkpoints/04_train_app_installments_cleaned.feather')
test_app_install = pd.read_feather('checkpoints/04_test_app_installments_cleaned.feather')
print('app_installments', train_app_install.shape, test_app_install.shape)

train_app_bureau = pd.read_feather('checkpoints/04_train_app_bureau_balance_bureau_cleaned.feather')
test_app_bureau = pd.read_feather('checkpoints/04_test_app_bureau_balance_bureau_cleaned.feather')
print('app_bureau', train_app_bureau.shape, test_app_bureau.shape)



In [ ]:
from functools import  reduce
def merge_dataframes(dfs, key):
    res = dfs[0].copy()
    # 不要合并重复的列
    for i,df in enumerate(dfs[1:], 1):
        unique_cols = [col for col in df.columns if col not in res.columns] + [key]
        res = pd.merge(res, df[unique_cols], on=key, how='left')
    return res

In [ ]:
train_dfs = [train_app_previous, train_app_credit, train_app_pos, train_app_install, train_app_bureau]
test_dfs = [test_app_previous, test_app_credit, test_app_pos, test_app_install, test_app_bureau]
train = merge_dataframes(train_dfs, key='SK_ID_CURR')
test = merge_dataframes(test_dfs, key='SK_ID_CURR')

In [ ]:
print(train.shape, test.shape)

In [ ]:
train.to_feather('checkpoints/05_train_merged_v1.feather')
test.to_feather('checkpoints/05_test_merged_v1.feather')


## modeling


In [ ]:
train = pd.read_feather('checkpoints/05_train_merged_v1.feather')
test = pd.read_feather('checkpoints/05_test_merged_v1.feather')

train_labels = train['TARGET']
train_ids = train['SK_ID_CURR']
test_ids = test['SK_ID_CURR']
train_features = train.drop(columns=['TARGET', 'SK_ID_CURR'])
test_features = test.drop(columns=['SK_ID_CURR'])

In [ ]:
print(train.shape, test.shape)

In [ ]:
list(train.columns)

###  hgbt

In [ ]:
%%time
from sklearn.ensemble import HistGradientBoostingClassifier

hist_gradient_boost_model= HistGradientBoostingClassifier(
    max_iter = 100, # 树个数
    learning_rate = 0.1,
    max_depth = 5,
)
hist_gradient_boost_model.fit(train_features, train_labels)

In [ ]:
train_prob = hist_gradient_boost_model.predict_proba(train_features)
plot_roc(train_labels, train_prob[:,1], 'hist gb')


In [ ]:
import time
import os

def submit(ids, pred, name, feature_count=None):
    """
    ids: 测试集的 SK_ID_CURR
    pred: 模型预测概率
    name: 你的实验备注 (如 'lgb_v1', 'baseline')
    feature_count: 可选，记录模型使用了多少个特征
    """
    # 1. 创建提交 DataFrame
    submit_df = pd.DataFrame({
        'SK_ID_CURR': ids,
        'TARGET': pred
    })

    # 2. 生成时间戳 (格式: 0213_1530)
    timestamp = time.strftime("%m%d_%H%M")
    
    # 3. 构造文件名
    # 格式: 0213_1530_lgb_v1_f542.csv
    f_str = f"_f{feature_count}" if feature_count else ""
    filename = f"{timestamp}_{name}{f_str}.csv"
    
    # 4. 确保保存目录存在 (可选)
    if not os.path.exists('submissions'):
        os.makedirs('submissions')
    
    save_path = os.path.join('submissions', filename)
    
    # 5. 保存并打印提示
    submit_df.to_csv(save_path, index=False)
    
    return submit_df


In [ ]:
submit_df = submit(test['SK_ID_CURR'], hist_gradient_boost_model_pred[:, 1], 
    name='hgbm_baseline',
    feature_count=train_features.shape[1]
    )
submit_df


得分 74， 有点不太合理

### lightbgm

In [ ]:
train_features_cleaned = clean_names(train_features)
test_features_cleaned = clean_names(test_features)

In [ ]:
%%time
from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(
    n_estimators=100,      # 对应 max_iter，树的个数
    learning_rate=0.1,     # 学习率
    max_depth=3,           # 树的最大深度
    random_state=42,       # 保证结果可复现
    n_jobs=-1              # 使用所有 CPU 核心加速
)
lgbm_model.fit(train_features_cleaned, train_labels)

In [ ]:
lgbm_model_pred = lgbm_model.predict_proba(test_features_cleaned)[:, 1]

In [ ]:
submit_df = submit(test['SK_ID_CURR'], lgbm_model_pred, 
    name='lgbm_baseline',
    feature_count=train_features.shape[1]
    )
submit_df


In [ ]:
features_importance = pd.DataFrame(
    {
        'importance': lgbm_model.feature_importances_,
        'feature': lgbm_model.feature_name_
    }
)

In [ ]:
def plot_features_importance(df):
    df = df.sort_values(by='importance', ascending=False).head(20)
    plt.figure(figsize=(10,6))
    sns.barplot(
        data = df,
        x= 'importance',
        y = 'feature'
    )
    plt.tight_layout()

In [ ]:
plot_features_importance(features_importance)

我们可以看到，我们的特征选择是错误的，没用的，新表没有特征上榜